In [1]:
import json
from typing import TypedDict
import os
from dotenv import load_dotenv

# Load variables from .env (if present) and fallback to existing environment variables
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY is not set. Add it to your environment or .env file.")

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is not set. Add it to your environment or .env file.")

In [ ]:
import dspy

# Connect to and set an LLM
lm = dspy.LM('anthropic/claude-haiku-4-5', api_key=ANTHROPIC_API_KEY, max_tokens=64000)
dspy.configure(lm=lm)

# Test
lm("Say hello")

['Hello! 👋 How can I help you today?']

In [7]:
from tavily import TavilyClient
from typing import Literal

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def internet_search(query: str, max_results: int = 5, include_raw_content: bool = False): 
	"""Run a web search""" 
	return tavily_client.search(
		query, max_results=max_results, 
		include_raw_content=include_raw_content, 
		topic="general",
	)

def read_webpage(url: str) -> dict:
	"""Read the content of a URL."""
	response = tavily_client.extract(url)
	return response

In [10]:
# Set our budget
BUDGETS = {
    "light":  {"max_subtopics": 3, "num_sources": 2, "max_search_results": 3},
    "medium": {"max_subtopics": 5, "num_sources": 2, "max_search_results": 5},
    "deep":   {"max_subtopics": 7, "num_sources": 5, "max_search_results": 10},
}
budget = BUDGETS["light"]

# Output destinations
ARTIFACT_OUTPUT = "output/research_artifact.json"
REPORT_OUTPUT = "output/workflow.md"

In [11]:
research_request = "Write a short explanation about Recrusive Language Models (RLMs)"


In [12]:
# Create our clarifier program
class ClarifyingSignature(dspy.Signature):
    "Generate clarifying questions to inform research work in response to the request."
    research_request: str = dspy.InputField()
    number_of_questions: int = dspy.InputField(desc="The number of clarifying questions to generate")
    clarifying_questions: list[str] = dspy.OutputField()

clarifier = dspy.Predict(ClarifyingSignature)

In [13]:
# Generate our questions
results = clarifier(research_request=research_request, number_of_questions=3)

# Get our answers
q_and_a = []
for question in results.clarifying_questions:
    answer = input(f"{question}\n")
    q_and_a.append({"clarifying_question": question, "user_guidance": answer})

In [14]:
q_and_a

[{'clarifying_question': "What is your target audience's level of expertise with language models and machine learning concepts?",
  'user_guidance': 'junior and senior engineers'},
 {'clarifying_question': 'Should the explanation focus on the theoretical foundations of RLMs, their practical applications, or both?',
  'user_guidance': 'both'},
 {'clarifying_question': 'Are there specific use cases or domains (e.g., code generation, mathematical reasoning, creative writing) you want the explanation to emphasize?',
  'user_guidance': 'accurate data retrieval in large context docs'}]

In [15]:
planner = dspy.Predict("research_request: str, clarifying_questions_and_answers, max_number_of_sub_topics: int -> topics_to_research: list[str]")

In [16]:
topics_to_research = planner(
    research_request=research_request,
    clarifying_questions_and_answers=q_and_a,
    max_number_of_sub_topics=budget["max_subtopics"],
).topics_to_research

print("\nSubtopics to research:")
for topic in topics_to_research:
    print(f"- {topic}")


Subtopics to research:
- Theoretical Foundations of Recursive Language Models: Architecture and Mechanisms
- Practical Applications of RLMs for Accurate Data Retrieval in Large Context Documents
- Comparative Analysis: RLMs vs Traditional Language Models for Engineering Teams


In [17]:
gatherer_sig = "research_request: str, subtopic_to_research: str, num_sources: int -> relevant_urls_to_investigate: list[str]"
gatherer = dspy.ReAct(gatherer_sig, tools=[internet_search], max_iters=budget["max_search_results"])

In [18]:
class Subtopic(TypedDict):
    subtopic: str
    urls: list[str]

In [19]:
subtopics_with_sources: list[Subtopic] = []
for subtopic in topics_to_research:
    urls = gatherer(
        research_request=research_request,
        subtopic_to_research=subtopic,
        num_sources=budget["num_sources"],
    ).relevant_urls_to_investigate
    subtopics_with_sources.append(Subtopic(subtopic=subtopic, urls=urls))

print("\nSources by subtopic:")
for subtopic_sources in subtopics_with_sources:
    print(f"Subtopic: {subtopic_sources['subtopic']}")
    for url in (subtopic_sources.get("urls") or []):
        print(f"- {url}")
    print()


Sources by subtopic:
Subtopic: Theoretical Foundations of Recursive Language Models: Architecture and Mechanisms
- https://alexzhang13.github.io/blog/2025/rlm/
- https://medium.com/@adeeba.noorr/recursive-language-models-the-architecture-that-thinks-in-layers-3c4c94da3cd8

Subtopic: Practical Applications of RLMs for Accurate Data Retrieval in Large Context Documents
- https://medium.com/@seemabanu1610/recursive-language-models-rlms-breaking-the-context-ceiling-in-ai-f92dfc291db4
- https://towardsdatascience.com/going-beyond-the-context-window-recursive-language-models-in-action/

Subtopic: Comparative Analysis: RLMs vs Traditional Language Models for Engineering Teams
- https://datalakehousehub.com/blog/2026-01-recursive-langauge-models/
- https://towardsdatascience.com/going-beyond-the-context-window-recursive-language-models-in-action/



In [20]:
class ProcessSource(dspy.Signature):
    """
    Process a source (e.g. a webpage) to extract relevant information that helps answer the research question.
    """
    research_task: str = dspy.InputField(desc="The overall research task or question that we are trying to answer. Use this as context when determining what information is relevant in the source.")
    research_subtask: str = dspy.InputField(desc="The subtopic that this source is meant to inform.")
    url: str = dspy.InputField()
    page_content: dict = dspy.InputField(desc="The content of the webpage; may also include metadata such as the title, author, publication date, etc.")
    summary: str = dspy.OutputField(descriptions="A concise summary of the source, focusing on information relevant to the research question. Aim for 1-2 paragraphs that capture the essence of the source's content in relation to the research question.")
    relevant_facts: list[str] = dspy.OutputField(descriptions="A list of relevant facts extracted from the source that help answer the research question.")
    interesting_annecdotes: list[str] = dspy.OutputField(description="Interesting or unique pieces of information that could make the report more engaging, relatable, or compelling.")
    additional_topics_to_explore: list[str] = dspy.OutputField(description="Additional items or questions to explore that are identified while processing the source.")

source_processing_agent = dspy.Predict(ProcessSource)

In [21]:
processed_sources = []
for subtopic_source in subtopics_with_sources:
    subtopic_findings = {"subtopic": subtopic_source['subtopic'], "findings": []}
    urls = subtopic_source.get("urls") or []
    for url in urls:
        if url.endswith('.pdf'):
            print(f"  Skipping PDF: {url}")
            continue
        page_content = read_webpage(url)
        result = source_processing_agent(
            research_task=research_request,
            research_subtask=subtopic_source["subtopic"],
            url=url,
            page_content=page_content,
        )
        subtopic_findings["findings"].append({
            "url": url,
            "summary": result.summary,
            "relevant_facts": result.relevant_facts,
            "interesting_annecdotes": result.interesting_annecdotes,
            "additional_topics_to_explore": result.additional_topics_to_explore,
        })
        print(f"Processed: {url}")
    processed_sources.append(subtopic_findings)

Processed: https://alexzhang13.github.io/blog/2025/rlm/
Processed: https://medium.com/@adeeba.noorr/recursive-language-models-the-architecture-that-thinks-in-layers-3c4c94da3cd8
Processed: https://medium.com/@seemabanu1610/recursive-language-models-rlms-breaking-the-context-ceiling-in-ai-f92dfc291db4
Processed: https://towardsdatascience.com/going-beyond-the-context-window-recursive-language-models-in-action/
Processed: https://datalakehousehub.com/blog/2026-01-recursive-langauge-models/
Processed: https://towardsdatascience.com/going-beyond-the-context-window-recursive-language-models-in-action/


In [22]:
if ARTIFACT_OUTPUT:
    os.makedirs(os.path.dirname(ARTIFACT_OUTPUT) or ".", exist_ok=True)
    with open(ARTIFACT_OUTPUT, "w") as f:
        json.dump(processed_sources, f, indent=4)
    print(f"\nResearch artifacts saved to {ARTIFACT_OUTPUT}")


Research artifacts saved to output/research_artifact.json


In [23]:
synthesizer = dspy.ChainOfThought(
    "research_request: str, clarifying_questions_and_answers, subtopic_research: list[dict] -> final_report: str"
)

In [24]:
final_report = synthesizer(
    research_request=research_request,
    clarifying_questions_and_answers=q_and_a,
    subtopic_research=processed_sources,
).final_report

print(final_report)

# Recursive Language Models (RLMs): A Practical Guide for Engineers

## Overview

Recursive Language Models (RLMs) are an inference strategy that fundamentally changes how language models interact with large contexts. Instead of feeding entire documents into a fixed context window, RLMs treat input as a programmable environment that the model can selectively explore and decompose through recursive function calls.

## The Problem They Solve: Context Rot

Traditional LLMs suffer from **context rot**—a documented performance degradation as context length increases, even within the model's nominal context window. In practice, a model's effective context length is often only ~50% of its advertised window. This makes handling thousands of documents or multi-million token contexts extremely challenging.

RLMs solve this by never forcing the entire input into attention at once. The model's actual input context window grows slowly even while handling massive overall context—up to two orders of 

In [25]:
class Annotator(dspy.Signature):
    """
    Add citations to a research report.

    Given a research report, go through each sentence and paragraph and determine
    if there is a corresponding claim in the processed sources that can be cited as evidence.

    If so, add a citation in the form of [source](source_url) after the claim.
    The output should be the final report with inline citations added.
    """
    final_report: str = dspy.InputField(desc="The report that needs to be annotated with citations.")
    processed_sources: list[dict] = dspy.InputField()
    annotated_report: str = dspy.OutputField()

annotator = dspy.Predict(Annotator)

In [26]:
result = annotator(
    final_report=final_report,
    processed_sources=processed_sources,
)

print(result.annotated_report)

# Recursive Language Models (RLMs): A Practical Guide for Engineers

## Overview

[Recursive Language Models (RLMs) are an inference strategy that fundamentally changes how language models interact with large contexts.](https://alexzhang13.github.io/blog/2025/rlm/) Instead of feeding entire documents into a fixed context window, [RLMs treat input as a programmable environment that the model can selectively explore and decompose through recursive function calls.](https://alexzhang13.github.io/blog/2025/rlm/)

## The Problem They Solve: Context Rot

[Traditional LLMs suffer from **context rot**—a documented performance degradation as context length increases, even within the model's nominal context window.](https://towardsdatascience.com/going-beyond-the-context-window-recursive-language-models-in-action/) [In practice, a model's effective context length is often only ~50% of its advertised window.](https://towardsdatascience.com/going-beyond-the-context-window-recursive-language-models-

In [27]:
with open(REPORT_OUTPUT, "w") as f:
    f.write(result.annotated_report)
print(f"\nFinal report with citations saved to {REPORT_OUTPUT}")


Final report with citations saved to output/workflow.md
